# 06 - Synthetic Control: Florence

For a single treated unit (Florence), construct a synthetic control by finding a weighted combination of control cities that best matches Florence's pre-treatment log_listings trajectory.

In [ ]:
# Data setup
# Set DATA_FILE to 'city_month_panel_real.parquet' after running build_real_panel.py
DATA_FILE = "city_month_panel_real.parquet"   # real Inside Airbnb data
# DATA_FILE = "city_month_panel.parquet"       # synthetic fallback (not committed)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

panel = pd.read_parquet(DATA_DIR / DATA_FILE)
panel["month"] = pd.to_datetime(panel["month"])

regs = pd.read_csv("../data/regulations.csv", parse_dates=["enforcement_date"])

print(f"Panel: {panel.shape}  |  Cities: {sorted(panel['city'].unique())}")
print(f"Date range: {panel['month'].min().date()} to {panel['month'].max().date()}")

In [ ]:
from scipy.optimize import minimize

## Setup: donor pool and pre/post windows

In [ ]:
FLORENCE_TREAT = pd.Timestamp("2023-10-01")
OUTCOME = "log_listings"

treated_city = "Florence"
donor_cities = ["Amsterdam","Lisbon","Vienna","Barcelona"]   # exclude NYC (also treated)

# Pivot to wide: city x month
wide = panel.pivot(index="month", columns="city", values=OUTCOME)
wide = wide.sort_index()

pre_months  = wide[wide.index < FLORENCE_TREAT].index
post_months = wide[wide.index >= FLORENCE_TREAT].index

Y_treated = wide.loc[:, treated_city].values
Y_donors  = wide.loc[:, donor_cities].values  # shape: (months, donors)

Y_treated_pre = wide.loc[pre_months, treated_city].values
Y_donors_pre  = wide.loc[pre_months, donor_cities].values

print(f"Pre-months:  {len(pre_months)}")
print(f"Post-months: {len(post_months)}")
print(f"Donors:      {donor_cities}")

## Solve for Abadie weights

In [ ]:
def synth_loss(w, Y_treated_pre, Y_donors_pre):
    """MSE between treated pre-period and synthetic control pre-period."""
    synth_pre = Y_donors_pre @ w
    return np.mean((Y_treated_pre - synth_pre) ** 2)

n_donors = len(donor_cities)
w0 = np.ones(n_donors) / n_donors
constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
bounds = [(0, 1)] * n_donors

result = minimize(synth_loss, w0, args=(Y_donors_pre, Y_donors_pre),
                  method="SLSQP", bounds=bounds, constraints=constraints)
# Note: correct argument order
result2 = minimize(synth_loss, w0, args=(Y_treated_pre, Y_donors_pre),
                   method="SLSQP", bounds=bounds, constraints=constraints)

weights = result2.x
for city, w in zip(donor_cities, weights):
    print(f"  {city:<15} weight = {w:.4f}")

pre_rmse = np.sqrt(synth_loss(weights, Y_treated_pre, Y_donors_pre))
print(f"\nPre-period RMSE: {pre_rmse:.4f}  (lower = better fit)")

## Synthetic control plot

In [ ]:
synth_series = Y_donors @ weights
months = wide.index

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(months, Y_treated,   lw=2.0, c="steelblue", label="Florence (actual)")
ax.plot(months, synth_series, lw=1.8, c="orange", ls="--", label="Synthetic control")
ax.axvline(FLORENCE_TREAT, color="red", ls="--", lw=1.2, label="Historic centre ban (Oct 2023)")
ax.fill_between(months, Y_treated, synth_series,
                where=(months >= FLORENCE_TREAT), alpha=0.15, color="red", label="Gap (ATT)")
ax.set_ylabel(f"log Listings")
ax.set_title("Synthetic Control: Florence vs Donor Pool\n(Abadie weights; pre-period RMSE shown in legend)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "06_synthetic_control_florence.png", bbox_inches="tight")
plt.show()

post_gap = Y_treated[len(pre_months):] - synth_series[len(pre_months):]
print(f"Post-treatment mean gap: {post_gap.mean():.4f}  (in log listings)")
print(f"Implied % change: {(np.exp(post_gap.mean())-1)*100:+.1f}%")

## In-space placebo tests

In [ ]:
# Apply synthetic control to each donor city, treating it as if it were treated
# Florence's gap should exceed most/all placebo gaps for the effect to be credible

placebo_gaps = {}
for placebo_city in donor_cities:
    other_donors = [c for c in donor_cities if c != placebo_city]
    Y_plac_pre   = wide.loc[pre_months, placebo_city].values
    Y_plac_all   = wide.loc[:, placebo_city].values
    Y_dplac_pre  = wide.loc[pre_months, other_donors].values
    Y_dplac_all  = wide.loc[:, other_donors].values

    res = minimize(synth_loss, np.ones(len(other_donors))/len(other_donors),
                   args=(Y_plac_pre, Y_dplac_pre),
                   method="SLSQP",
                   bounds=[(0,1)]*len(other_donors),
                   constraints=[{"type":"eq","fun":lambda w: w.sum()-1}])
    synth_plac = Y_dplac_all @ res.x
    placebo_gaps[placebo_city] = (Y_plac_all[len(pre_months):] - synth_plac[len(pre_months):]).mean()

florence_gap = post_gap.mean()
print("Post-treatment gaps:")
print(f"  Florence (treated): {florence_gap:.4f}")
for city, gap in placebo_gaps.items():
    print(f"  {city:<15} (placebo): {gap:.4f}")

rank = sum(1 for g in placebo_gaps.values() if abs(g) >= abs(florence_gap))
print(f"\nFlorence gap exceeds {len(placebo_gaps)-rank}/{len(placebo_gaps)} placebo gaps in magnitude")